# Notebook 01: Markets, Prices, Returns, and Point-in-Time Data

**Phase 1 — Foundations (`QUANT_PROJECT_DESIGN.md` §1.2)**

### Learning Objectives
1. Understand US equity/ETF market structure (`XNYS` calendar, regular hours `09:30–16:00 America/New_York`) and our survivorship-free Tier-1 US ETF universe.
2. Enforce bitemporal Point-in-Time (PIT) discipline (`period_ts` vs. `available_ts`) across EST/EDT transitions.
3. Contrast naive backward-adjusted or unadjusted prices against **immutable raw OHLCV + explicit corporate actions**.
4. Verify return conversions (simple vs. log returns, time vs. cross-sectional aggregation) and catch `lag=0` look-ahead bias.


## 1. Concept & CS Analogy: Append-Only Event Logs vs. Mutable History

- **Financial Concept:** A daily US equity bar for date $d$ becomes knowable only at the market close (`16:00:00 America/New_York`). Splits and dividends occur on `ex_date` and alter share counts or cash balances without rewriting past executed trade prices.
- **CS Analogy (Immutable Event Log & Bitemporal Indexing):**
  - Store **raw OHLCV** as an immutable append-only log keyed by `(ticker, period_ts)`, paired with an append-only `actions` log `(ticker, ex_date, split_factor, div_cash)`.
  - Backward-adjusted `Adj Close` mutates *every historical record* whenever a new dividend is paid—breaking SHA-256 content hashes, historical option strike comparisons, and reproducibility.
  - Every record carries two UTC timestamps: `period_ts` (event date at `00:00:00Z`) and `available_ts` (exact commit/knowledge time in UTC: `21:00:00Z` in EST winter, `20:00:00Z` in EDT summer).


In [1]:
from datetime import date

from quantlab.data import (
    TIER1_US_ETFS,
    date_to_period_ts,
    us_equity_close_available_ts,
)

print(f"Tier-1 US Market & Sector ETFs ({len(TIER1_US_ETFS)}):", TIER1_US_ETFS)

winter_d = date(2025, 1, 15)  # EST (UTC-5)
summer_d = date(2025, 7, 15)  # EDT (UTC-4)

for label, d in [("Winter EST", winter_d), ("Summer EDT", summer_d)]:
    p_ts = date_to_period_ts(d)
    a_ts = us_equity_close_available_ts(d)
    print(f"{label} | period_ts={p_ts.isoformat()} | available_ts={a_ts.isoformat()}")

assert us_equity_close_available_ts(winter_d).hour == 21
assert us_equity_close_available_ts(summer_d).hour == 20

Tier-1 US Market & Sector ETFs (14): ('SPY', 'QQQ', 'IWM', 'XLK', 'XLF', 'XLV', 'XLE', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC')
Winter EST | period_ts=2025-01-15T00:00:00+00:00 | available_ts=2025-01-15T21:00:00+00:00
Summer EDT | period_ts=2025-07-15T00:00:00+00:00 | available_ts=2025-07-15T20:00:00+00:00


## 2. Mathematical Foundations: Simple vs. Log Returns & Corporate Actions

1. **Simple Price Return (no corporate action):**
   $$R_t = \frac{P_t}{P_{t-1}} - 1$$
2. **Corporate-Action Total Simple Return** (with split factor $S_t$ new shares per old share, and cash dividend $D_t$ USD per pre-split share):
   $$R_t^{\text{tot}} = \frac{P_t^{\text{raw}} \cdot S_t + D_t}{P_{t-1}^{\text{raw}}} - 1$$
3. **Log (Continuously Compounded) Return:**
   $$r_t = \ln(1 + R_t) \quad \Longleftrightarrow \quad R_t = \exp(r_t) - 1$$
4. **Two Non-Negotiable Aggregation Rules (`docs/conventions.md` §2.3):**
   - **Time axis:** Log returns add across time ($r_{t_0 \to t_k} = \sum_{s=1}^k r_{t_s}$); simple returns multiply ($\prod_s (1 + R_{t_s}) - 1$).
   - **Asset cross-section:** Portfolio returns are weighted sums of **simple** returns ($R_{p,t} = \sum_{i=1}^N w_{i,t-1} R_{i,t}$). Never take a weighted sum of log returns across assets!


In [2]:
import numpy as np

from quantlab.core import compound_returns, log_to_simple_returns, simple_to_log_returns
from quantlab.data import (
    compute_total_return_index,
    compute_total_returns,
    make_toy_5bar_fixture,
)

bars_toy, actions_toy = make_toy_5bar_fixture("TOY")
tri_df = compute_total_return_index(compute_total_returns(bars_toy, actions=actions_toy))

print("5-Bar Corporate Action Fixture (2-for-1 split on Bar 2, $1.00 dividend on Bar 3):")
for row in tri_df.select(
    ["period_ts", "close", "split_factor", "div_cash", "price_return", "total_return", "tri"]
).iter_rows(named=True):
    print(
        f"  {row['period_ts'].date()} | close={row['close']:6.2f} | "
        f"split={row['split_factor']:.1f} | div=${row['div_cash']:.2f} | "
        f"naive_price_ret={row['price_return']:+7.3f} | "
        f"total_ret={row['total_return']:+7.3f} | TRI={row['tri']:6.2f}"
    )

tot_rets = tri_df["total_return"].to_numpy()[1:]
log_rets = simple_to_log_returns(tot_rets)
round_trip = log_to_simple_returns(log_rets)
cum_simple = compound_returns(tot_rets)
cum_from_log = float(np.exp(np.sum(log_rets)) - 1.0)

assert np.allclose(round_trip, tot_rets)
assert np.isclose(cum_simple, 0.21)
assert np.isclose(cum_simple, cum_from_log)
print(f"Cumulative return: simple={cum_simple:.4f}, from summed log={cum_from_log:.4f}")

5-Bar Corporate Action Fixture (2-for-1 split on Bar 2, $1.00 dividend on Bar 3):
  2025-01-06 | close=100.00 | split=1.0 | div=$0.00 | naive_price_ret=   +nan | total_ret=   +nan | TRI=100.00
  2025-01-07 | close=110.00 | split=1.0 | div=$0.00 | naive_price_ret= +0.100 | total_ret= +0.100 | TRI=110.00
  2025-01-08 | close= 55.00 | split=2.0 | div=$0.00 | naive_price_ret= -0.500 | total_ret= +0.000 | TRI=110.00
  2025-01-09 | close= 54.00 | split=1.0 | div=$1.00 | naive_price_ret= -0.018 | total_ret= +0.000 | TRI=110.00
  2025-01-10 | close= 59.40 | split=1.0 | div=$0.00 | naive_price_ret= +0.100 | total_ret= +0.100 | TRI=121.00
Cumulative return: simple=0.2100, from summed log=0.2100


## 3. Point-in-Time (PIT) Parquet Store, Validation, and Performance Metrics

Rather than hitting live vendor APIs in tests, `SyntheticDataProvider` produces deterministic multi-asset US equity panels with shared market factor structure and synthetic corporate actions. Every dataset is validated by `validate_bars`, content-hashed with SHA-256, and queried strictly with `available_ts <= as_of_ts`.


In [3]:
import tempfile
from pathlib import Path

from quantlab.core import summarize_performance
from quantlab.data import (
    PointInTimeStore,
    SyntheticDataProvider,
    us_equity_close_available_ts,
    validate_bars,
)

tickers = ["SPY", "QQQ", "IWM", "XLK", "XLF"]
start_d, end_d = date(2024, 1, 2), date(2024, 12, 31)

provider = SyntheticDataProvider(seed=42, annual_drift=0.08, annual_vol=0.18)
bars = provider.fetch_bars(tickers, start_d, end_d)
actions = provider.fetch_corporate_actions(tickers, start_d, end_d)

report = validate_bars(bars, actions=actions)
report.raise_if_invalid()
print(f"Validation passed: {report.row_count} bars, {report.ticker_count} tickers.")

with tempfile.TemporaryDirectory() as tmpdir:
    store = PointInTimeStore(Path(tmpdir))
    content_hash = store.ingest(bars, actions)
    print(f"PIT Store SHA-256: {content_hash[:16]}...")

    # Query strictly as of mid-year close
    mid_cutoff = us_equity_close_available_ts(date(2024, 6, 28))
    mid_bars = store.query_bars(mid_cutoff)
    assert mid_bars["available_ts"].max() <= mid_cutoff

    # Extract full-year (T, N) float64 total-return panel at year-end cutoff
    year_end_cutoff = us_equity_close_available_ts(end_d)
    rets_2d, period_ts_list, sorted_tickers = store.to_return_panel(year_end_cutoff)

print(f"Return panel shape: {rets_2d.shape} (T={len(period_ts_list)}, N={len(sorted_tickers)})")
print("Tickers (sorted):", sorted_tickers)

for col_idx, tkr in enumerate(sorted_tickers):
    perf = summarize_performance(rets_2d[:, col_idx])
    print(
        f"  {tkr}: CAGR={perf.cagr:+6.2%} | Vol={perf.annualized_vol:5.2%} | "
        f"Sharpe={perf.sharpe:+5.2f} (SE={perf.sharpe_se:.2f}) | MDD={perf.max_drawdown:6.2%}"
    )

Validation passed: 1305 bars, 5 tickers.
PIT Store SHA-256: 2dd1dc3aa73d94d3...
Return panel shape: (260, 5) (T=260, N=5)
Tickers (sorted): ['IWM', 'QQQ', 'SPY', 'XLF', 'XLK']
  IWM: CAGR=-14.79% | Vol=17.98% | Sharpe=-0.80 (SE=0.99) | MDD=24.74%
  QQQ: CAGR=-3.96% | Vol=17.21% | Sharpe=-0.15 (SE=0.99) | MDD=17.93%
  SPY: CAGR=+3.89% | Vol=17.48% | Sharpe=+0.31 (SE=0.99) | MDD=11.75%
  XLF: CAGR=-6.99% | Vol=18.32% | Sharpe=-0.30 (SE=0.99) | MDD=13.72%
  XLK: CAGR=-18.50% | Vol=17.61% | Sharpe=-1.07 (SE=0.98) | MDD=31.23%


## 4. Two Classic Pitfalls & Verification Exercise

1. **Pitfall A — Jensen's Inequality Error on Portfolio Log Returns:** Computing $\exp(\sum_i w_i r_{i,t}) - 1$ instead of $\sum_i w_i R_{i,t}$ systematically understates diversified portfolio returns because $\ln(\cdot)$ is strictly concave ($\mathbb{E}[\ln X] < \ln \mathbb{E}[X]$).
2. **Pitfall B — Contemporaneous (`lag=0`) Look-Ahead Bias:** Forming weights $w_t = f(\mathcal{F}_t)$ at `close[t]` and multiplying directly by $R_t$ (`lag=0`) instead of `shift(+1)` (`w[t-1] * R[t]`) fabricates massive spurious Sharpe ratios.


In [4]:
from quantlab.core import portfolio_returns, sharpe_ratio
from quantlab.validation import evaluate_signal_lag_and_shift_probe

# Pitfall A: True cross-sectional simple-return portfolio vs. naive log-return average
n_assets = rets_2d.shape[1]
w_eq = np.full(n_assets, 1.0 / n_assets, dtype=np.float64)

true_port_ret = portfolio_returns(rets_2d, w_eq)
wrong_port_ret = np.exp(simple_to_log_returns(rets_2d) @ w_eq) - 1.0

true_cum = compound_returns(true_port_ret)
wrong_cum = compound_returns(wrong_port_ret)
diff_bps = (true_cum - wrong_cum) * 1e4
print(
    f"Pitfall A (1/N over {rets_2d.shape[0]} days): "
    f"true cum={true_cum:.6f}, naive log-sum cum={wrong_cum:.6f} "
    f"(error = {diff_bps:.2f} bps)"
)
assert true_cum > wrong_cum

# Pitfall B: Shift probe on SPY daily sign signal w[t] = sign(r[t])
spy_rets = rets_2d[:, sorted_tickers.index("SPY")]
probe = evaluate_signal_lag_and_shift_probe(lambda r: np.sign(r), spy_rets, spy_rets)
print("Pitfall B (Shift Probe on w[t] = sign(r[t])):")
print(f"  Causal lag=+1 Sharpe       : {probe.metadata['sharpe_causal_lag1']:+6.2f}")
print(f"  Delayed lag=+2 Sharpe      : {probe.metadata['sharpe_delayed_lag2']:+6.2f}")
print(f"  ILLEGAL lag=0 Sharpe       : {probe.metadata['sharpe_contemporaneous_lag0']:+6.2f}")
print(f"  Peek lag=-1 Sharpe         : {probe.metadata['sharpe_peek_minus1']:+6.2f}")

assert probe.passed
assert probe.metadata["sharpe_contemporaneous_lag0"] > 5.0
assert float(sharpe_ratio(true_port_ret)) == summarize_performance(true_port_ret).sharpe
print("All Notebook 01 checks passed!")

Pitfall A (1/N over 260 days): true cum=-0.080616, naive log-sum cum=-0.086675 (error = 60.59 bps)
Pitfall B (Shift Probe on w[t] = sign(r[t])):
  Causal lag=+1 Sharpe       :  +0.39
  Delayed lag=+2 Sharpe      :  -1.04
  ILLEGAL lag=0 Sharpe       : +21.98
  Peek lag=-1 Sharpe         :  +0.44
All Notebook 01 checks passed!
